# OWL: wildlife localization in overhead imagery

Run a pretrained **OWL-D** model on four 512 × 512 sample patches, inspect predicted animal locations, and save reusable results. Two 5472 × 3648 survey images are an **optional tiled-inference** step. No annotations, training, species identification, or accuracy evaluation are required or provided.

**Default model:** the released `OWL-D.pth` is **`OWLD_H`**, with a frozen DINOv3 ViT-H+/16 backbone and DPT decoder. Its approximately **3.5 GB** checkpoint contains the backbone; inference constructs the model with `pretrained=False` and does **not** require a separate Meta download. Allow disk space for the environment, cached checkpoint, and generated figures, plus sufficient host RAM and CUDA memory. The environment stage reports resources before sample/model downloads. The supported OWL-D notebook route requires a compatible NVIDIA CUDA GPU. For CPU, explicitly set `MODEL = 'owl-c'` and `DEVICE = 'cpu'`; this selects the released general OWL-C checkpoint, never a silent substitute.

**Local execution verified (2026-09-08):** this 23-cell notebook completed through nbclient with OWL-D on all four patches and both full-resolution images, and a separate explicitly configured OWL-C CPU notebook run completed. Executed outputs include environment information, progress, tables, and inline figures. The source notebook keeps its outputs cleared; local executed copies and reproduction commands are linked at the end.

**Publication location:** this notebook belongs in the existing [microsoft/MegaDetector-Overhead](https://github.com/microsoft/MegaDetector-Overhead) repository at `notebooks/owl_inference_demo.ipynb`, together with its backend and manifest. GitHub previews the file; the supported first-release workflow runs locally in Jupyter or VS Code. The sample ZIP will be a GitHub release asset on the same repository, outside Git history; model weights remain on Zenodo. No upload or publication is performed by this notebook.

**Sample access:** the exact ZIP is available from [release owl-notebook-v1](https://github.com/microsoft/MegaDetector-Overhead/releases/tag/owl-notebook-v1). Preparation downloads it automatically using `notebooks/sample_data.json`, which records every sample's source and terms: contributor-confirmed SheepCounter **Public Domain** and HerdNet **CC BY-NC-SA 4.0**, plus primary-source-verified caribou **CC BY-NC-SA 4.0**. The mixed archive has no blanket software license. Set `ARCHIVE` if you already have the exact ZIP locally. Colab remains **experimental and unverified on a hosted runtime**; it does not block the supported local release.

The notebook kernel only uses Python's standard library and IPython for presentation. All model work runs in the project's `.venv` interpreter, so a hosted kernel outside the project's Python `>=3.11,<3.13` range does not import PyTorch or `animaloc`.

## 1. Configure the run

After the one-time environment setup, the defaults download the sample ZIP and model as needed and run four patches. Paths below are relative to the repository root unless absolute. Leave `SAMPLE_URL` empty to use the published URL in the manifest; `ARCHIVE` selects an existing local copy of the same ZIP. `CHECKPOINT` may point to an existing trusted checkpoint for the selected model; otherwise the backend uses the pinned Zenodo release and its validated cache.

For the experimental Colab path, first select a compatible **GPU runtime** for OWL-D and set `REPO_REF = 'owl-notebook-v1'`. This uses the matching notebook, backend, and manifest. Local Jupyter/VS Code setup is described in the next section.

In [ ]:
from pathlib import Path

MODEL = 'owl-d'                 # Explicit CPU opt-in: 'owl-c'.
DEVICE = 'auto'                 # 'auto', 'cuda', or 'cpu'; OWL-D requires CUDA.
SAMPLE_URL = ''                 # Empty uses the public release URL in the manifest.
ARCHIVE = None                  # Optional local copy of the exact sample ZIP.
CHECKPOINT = None               # None uses demo_data/models/OWL-D.pth or OWL-C.pth.
MODEL_CACHE = None              # Optional isolated cache; do not combine with CHECKPOINT.
DATA_DIR = Path('demo_data/owl_notebook')
RUNS_DIR = DATA_DIR / 'runs'

RUN_FULL_RESOLUTION = False     # Opt in to both full images, with tiling, not resizing.
RUN_CUSTOM_IMAGES = False       # Separate opt-in; does not replace the sample run.
CUSTOM_IMAGES_DIR = None        # Set your image directory when enabling custom inference.

SCORE_THRESHOLD = 0.2           # Absolute retained-peak score cutoff.
ADAPT_THRESHOLD = 0.3           # LMDS adaptive peak-selection threshold.
NEGATIVE_THRESHOLD = 0.1        # LMDS negative/background threshold.
TILE_SIZE = 512
OVERLAP = 160

REPO_PATH = None                # Local checkout override; otherwise discover from cwd.
REPO_URL = 'https://github.com/microsoft/MegaDetector-Overhead.git'
REPO_REF = ''                   # For a new Colab clone, set 'owl-notebook-v1'.
COLAB_CHECKOUT = Path.cwd() / 'MegaDetector-Overhead-notebook'
RESYNC_COLAB_ENVIRONMENT = False  # Explicit repair/update of this checkout's .venv only.


## 2. Locate the checkout and project Python

### Local Jupyter or VS Code

You do not need the author's Conda environments. Install [uv](https://docs.astral.sh/uv/getting-started/installation/) and Git, then clone the published repository containing this notebook and backend. `uv sync --locked` obtains the supported Python version and creates the checkout's own `.venv` from `.python-version`, `pyproject.toml`, and `uv.lock`. This includes Jupyter and the local `animaloc`/`dinov3` packages. The GPU route requires a compatible NVIDIA GPU and driver; uv installs the CUDA-enabled PyTorch wheels, not the system driver. This is a one-time terminal setup, not an automatic installer inside the local notebook:

```bash
git clone https://github.com/microsoft/MegaDetector-Overhead.git
cd MegaDetector-Overhead
# NVIDIA GPU / default OWL-D route:
uv sync --locked --no-default-groups --group gpu --extra notebook
# CPU route instead (also set MODEL='owl-c', DEVICE='cpu' above):
# uv sync --locked --extra notebook
.venv/bin/python -m ipykernel install --user --name megadetector-overhead --display-name 'MegaDetector Overhead'
.venv/bin/python -m jupyterlab notebooks/owl_inference_demo.ipynb
```

If the current project environment is already installed, launch Jupyter directly; no re-sync is needed. Choose **Restart Kernel and Run All Cells** for a clean four-patch run. Set `RUN_FULL_RESOLUTION = True` first to include the two large images. In VS Code, open the notebook and select **MegaDetector Overhead** or the checkout's `.venv` Python kernel, then restart and run all cells. On Windows, use `.venv\Scripts\python.exe` in place of `.venv/bin/python`. The setup cell **does not sync or replace a local environment**; it explains how to repair missing dependencies. Avoid bare `uv run` or `uv sync` after a GPU sync: the default group installs CPU PyTorch.

### Colab setup path — hosted validation pending

Open/upload this notebook in Colab, select the runtime, and set `REPO_REF = 'owl-notebook-v1'`. Setup clones **only if its dedicated destination is absent**. It validates an existing checkout and never resets it, pulls over changes, or overwrites its files. To use an existing checkout elsewhere, set `REPO_PATH` explicitly.

When needed, Colab installs `uv` with pip into its disposable hosted environment, then uses `uv sync --python 3.11` to create a supported **backend** interpreter. It does not try to replace the hosted notebook kernel or weaken the project's Python requirements. Sample preparation uses the public manifest URL, or an explicitly configured local `ARCHIVE`.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
if MODEL not in {'owl-d', 'owl-c'} or DEVICE not in {'auto', 'cpu', 'cuda'}:
    raise ValueError('Choose MODEL owl-d|owl-c and DEVICE auto|cpu|cuda.')
if MODEL == 'owl-d' and DEVICE == 'cpu':
    raise ValueError("OWL-D requires CUDA here. For CPU explicitly set MODEL='owl-c' and DEVICE='cpu'.")

if REPO_PATH is not None:
    REPO_ROOT = Path(REPO_PATH).expanduser().resolve()
elif IN_COLAB:
    REPO_ROOT = Path(COLAB_CHECKOUT).expanduser().resolve()
else:
    REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / 'pyproject.toml').is_file() and (p / '.git').exists()), None)
    if REPO_ROOT is None:
        raise RuntimeError('Open this notebook from the repository, or set REPO_PATH to your checkout.')

def git_output(*args):
    return subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=True,
                          capture_output=True, text=True).stdout.strip()

def resolve_ref(ref):
    for candidate in (ref, 'origin/' + ref):
        result = subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'rev-parse', '--verify', '--end-of-options', candidate + '^{commit}'],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'Repository ref {ref!r} is not available in this checkout. Fetch it explicitly first.')

if IN_COLAB and not REPO_ROOT.exists():
    if REPO_PATH is not None:
        raise FileNotFoundError('REPO_PATH must be an existing checkout; cloning uses COLAB_CHECKOUT only.')
    if not REPO_REF.strip():
        raise RuntimeError('Set REPO_REF to a published ref containing the notebook/backend; public main is not assumed ready.')
    subprocess.run(['git', 'clone', '--no-checkout', '--', REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', resolve_ref(REPO_REF)], check=True)

if not REPO_ROOT.is_dir() or not (REPO_ROOT / '.git').exists():
    raise RuntimeError(f'{REPO_ROOT} is not a Git checkout. Nothing was overwritten; choose another location.')
if Path(git_output('rev-parse', '--show-toplevel')).resolve() != REPO_ROOT:
    raise RuntimeError('REPO_PATH must point to the root of the project checkout.')
if IN_COLAB:
    origin = git_output('remote', 'get-url', 'origin').rstrip('/').removesuffix('.git')
    if origin != REPO_URL.rstrip('/').removesuffix('.git'):
        raise RuntimeError('Existing checkout origin differs from REPO_URL; configure the intended URL or another directory.')
if REPO_REF and git_output('rev-parse', 'HEAD') != resolve_ref(REPO_REF):
    raise RuntimeError('Existing checkout is not at REPO_REF. Select the ref yourself; setup never resets an existing checkout.')
required_files = ['pyproject.toml', 'uv.lock', 'dinov3/setup.py',
                  'tools/demo_owl_notebook.py', 'notebooks/sample_data.json',
                  'notebooks/owl_inference_demo.ipynb']
missing = [name for name in required_files if not (REPO_ROOT / name).is_file()]
if missing:
    raise RuntimeError(f'Checkout lacks required demo files: {missing}. Use the development/published notebook ref; no automatic update was made.')

def repo_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else REPO_ROOT / path).resolve()

DATA_DIR = repo_path(DATA_DIR)
RUNS_DIR = repo_path(RUNS_DIR)
BACKEND_PYTHON = REPO_ROOT / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
BACKEND_SCRIPT = REPO_ROOT / 'tools/demo_owl_notebook.py'
probe_code = (
    "import sys, importlib.util; "
    "assert (3, 11) <= sys.version_info[:2] < (3, 13), 'Backend requires Python >=3.11,<3.13'; "
    "missing = [m for m in ['torch', 'torchvision', 'animaloc', 'dinov3', 'PIL', 'pandas', 'matplotlib'] "
    "if importlib.util.find_spec(m) is None]; "
    "assert not missing, 'Missing backend packages: ' + ', '.join(missing)"
)

def backend_ready():
    if not BACKEND_PYTHON.is_file():
        return False
    return subprocess.run([str(BACKEND_PYTHON), '-c', probe_code], cwd=REPO_ROOT,
                          capture_output=True, text=True).returncode == 0

if IN_COLAB and (not backend_ready() or RESYNC_COLAB_ENVIRONMENT):
    gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                               capture_output=True, text=True) if shutil.which('nvidia-smi') else None
    has_gpu = gpu_probe is not None and gpu_probe.returncode == 0 and bool(gpu_probe.stdout.strip())
    if (MODEL == 'owl-d' or DEVICE == 'cuda') and not has_gpu:
        raise RuntimeError("Enable a compatible GPU runtime, or explicitly set MODEL='owl-c', DEVICE='cpu'. No model was substituted.")
    use_gpu = DEVICE != 'cpu' and (MODEL == 'owl-d' or DEVICE == 'cuda' or has_gpu)
    uv_executable = shutil.which('uv')
    if uv_executable:
        uv_command = [uv_executable]
    else:
        if importlib.util.find_spec('uv') is None:
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
        uv_command = [sys.executable, '-m', 'uv']
    subprocess.run(uv_command + ['sync', '--locked', '--python', '3.11', '--no-default-groups',
                                '--group', 'gpu' if use_gpu else 'cpu', '--extra', 'notebook'],
                   check=True, cwd=REPO_ROOT)

if not backend_ready():
    gpu_flags = ' --no-default-groups --group gpu' if MODEL == 'owl-d' or DEVICE != 'cpu' else ''
    raise RuntimeError(f'Project backend missing/incompatible. From {REPO_ROOT}, run:\n'
                       f'uv sync --locked{gpu_flags} --extra notebook\n'
                       'Then rerun setup. Local setup never re-syncs automatically.')

from IPython.display import FileLink, HTML, Image, display

def run_stage(stage, *arguments):
    command = [str(BACKEND_PYTHON), '-u', str(BACKEND_SCRIPT), stage, *map(str, arguments)]
    print(f'Running backend stage: {stage}', flush=True)
    subprocess.run(command, check=True, cwd=REPO_ROOT)

print('Notebook host:', sys.executable, sys.version.split()[0])
print('Project backend:', BACKEND_PYTHON)
print('Repository:', REPO_ROOT)
print('Repository HEAD:', git_output('rev-parse', 'HEAD'))
print('Mode:', 'Colab setup path (hosted validation pending)' if IN_COLAB else 'local; existing environment reused')
print('Selected model/device:', MODEL, DEVICE)


## 3. Check the backend before downloading assets

This stage inspects the project interpreter, repository, PyTorch/CUDA, GPU resources, and chosen model/device; **it does not download weights or samples**. OWL-D without working CUDA is an error, not an empty successful run. Enable a compatible GPU or deliberately select OWL-C for CPU, then rerun from configuration. The locked CUDA 12.1 group supports Volta through Hopper; GPUs requiring newer CUDA wheels need an explicitly configured compatible environment (see `INSTALL.md`).

In [ ]:
run_stage('environment', '--model', MODEL, '--device', DEVICE)


## 4. Prepare and inspect all six sample images

The backend checks the archive and inventory against `notebooks/sample_data.json`, safely extracts beneath `demo_data/owl_notebook/OWL_DATA`, and reuses verified local files on reruns. The supplied archive is approximately **18.3 MB**: four 512 × 512 patches and two 5472 × 3648 images, with **no ground-truth annotations**. It also writes a contact sheet and inventory CSV.

With no override, the manifest's published ZIP is downloaded to the data directory and verified/reused on later runs. `ARCHIVE` explicitly selects a local ZIP instead. A download or integrity failure stops preparation with an actionable error. The separate caribou dataset used by the shell demos is **not** a substitute for this exact sample archive.

In [ ]:
prepare_arguments = ['--data-dir', DATA_DIR]
if ARCHIVE is not None:
    prepare_arguments += ['--archive', repo_path(ARCHIVE)]
if SAMPLE_URL.strip():
    prepare_arguments += ['--sample-url', SAMPLE_URL.strip()]
run_stage('prepare', *prepare_arguments)
SAMPLE_IMAGES_DIR = DATA_DIR / 'OWL_DATA'


In [ ]:
import csv
import html
import itertools

def show_csv(path, limit=10):
    with Path(path).open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        columns = reader.fieldnames or []
        rows = list(itertools.islice(reader, limit))
    header = ''.join('<th>' + html.escape(column) + '</th>' for column in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(row.get(column, ''))) + '</td>'
                                  for column in columns) + '</tr>' for row in rows)
    display(HTML('<div style="overflow-x:auto"><table><caption>' + html.escape(Path(path).name)
                 + f' (first {limit} rows at most)</caption><thead><tr>' + header
                 + '</tr></thead><tbody>' + body + '</tbody></table></div>'))
    if not rows:
        print('No rows; CSV column headers are preserved.')

print('All six source images:', SAMPLE_IMAGES_DIR)
print('Default inference: four 512 × 512 patches. Optional inference: two 5472 × 3648 images.')
show_csv(DATA_DIR / 'sample_inventory.csv', limit=6)
display(Image(filename=str(DATA_DIR / 'contact_sheet.png'), width=1100))


## 5. Run the four patches

The backend validates/reuses `demo_data/models/OWL-D.pth` (or `OWL-C.pth` for explicit CPU opt-in), downloading the selected pinned public Zenodo file if absent. Network failures trigger bounded retries and an official alternate Zenodo endpoint; both endpoints use the same pinned size and SHA-256, and integrity failures are never bypassed. `MODEL_CACHE` can select a separate cache for clean-download checks without disturbing the working files; do not combine it with `CHECKPOINT`. The file sizes and SHA-256 pins are independently confirmed by the [published release README](https://zenodo.org/api/records/20802844/files/README.md/content). The release declares **CC BY-NC-SA 4.0**; also review the [DINOv3 terms](../dinov3/LICENSE.md) for the included backbone. It reports checkpoint loading, settings, resource information, per-image timing, and retained counts.

Defaults use 512-pixel tiles, 160-pixel overlap, mean stitching, half-resolution heatmaps, and LMDS peak extraction with a 3 × 3 kernel. `ADAPT_THRESHOLD` controls adaptive peak selection; `NEGATIVE_THRESHOLD` suppresses weak/background responses; `SCORE_THRESHOLD` is the separate absolute cutoff on retained peak scores. The exported rows, points, and counts use the same final detections.

Each inference call gets a new timestamp/UUID output directory. Rerunning a cell cannot accidentally display results from a previous model or threshold setting. Errors, including download failure, incompatible checkpoints, or out-of-memory, are not converted to zero predictions.

In [ ]:
from datetime import datetime, timezone
import uuid

def run_inference(images_dir, selection, label):
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output_dir = RUNS_DIR / f'{stamp}_{MODEL}_{label}_{uuid.uuid4().hex[:8]}'
    if output_dir.exists():
        raise FileExistsError(f'Refusing to reuse inference output directory: {output_dir}')
    arguments = ['--model', MODEL, '--device', DEVICE, '--images-dir', images_dir,
                 '--selection', selection, '--output-dir', output_dir,
                 '--score-threshold', SCORE_THRESHOLD, '--adapt-threshold', ADAPT_THRESHOLD,
                 '--negative-threshold', NEGATIVE_THRESHOLD, '--tile-size', TILE_SIZE, '--overlap', OVERLAP]
    if CHECKPOINT is not None:
        arguments += ['--checkpoint', repo_path(CHECKPOINT)]
    if MODEL_CACHE is not None:
        arguments += ['--model-cache', repo_path(MODEL_CACHE)]
    print(f'Model={MODEL}; device={DEVICE}; selection={selection}; images={images_dir}', flush=True)
    print(f'Tile={TILE_SIZE}; overlap={OVERLAP}; score={SCORE_THRESHOLD}; '
          f'adaptive={ADAPT_THRESHOLD}; negative={NEGATIVE_THRESHOLD}', flush=True)
    print('New output directory:', output_dir, flush=True)
    run_stage('infer', *arguments)
    return output_dir

PATCH_OUTPUT = None
PATCH_OUTPUT = run_inference(SAMPLE_IMAGES_DIR, 'patches', 'patches')


## 6. Read the results

Each comparison figure shows **original image → FIDT localization heatmap → predicted-point overlay**. Heatmap colors are normalized for display, not calibrated between images. Predicted animal count is the number of retained localization peaks. **FIDT map sums are not animal counts, and peak scores are not calibrated probabilities.**

`detections.csv` contains `images,x,y,dscores,labels`: `(x, y)` are already in **original-image pixels**. Do not apply an additional factor of two or the legacy evaluation visualizer's `--pred-scale 2`. Labels denote generic animals, not species. `summary.csv` contains `images,width,height,count,seconds,tiles`, including a row with `count=0` for a valid image without predictions. An empty detections CSV still has its headers.

The total below is a sum across demo images, **not a population estimate**: patches and full images may overlap or contain the same animals. These samples have no annotations, so there is no precision, recall, F1, or accuracy claim. Visually inspect missed animals, false positives, and tile boundaries.

In [ ]:
import json

def show_run(output_dir, expected_images=None):
    if output_dir is None:
        raise RuntimeError('No completed run is available. Execute its inference cell successfully first.')
    with (output_dir / 'summary.csv').open(newline='', encoding='utf-8') as handle:
        summary = list(csv.DictReader(handle))
    if expected_images is not None and len(summary) != expected_images:
        raise RuntimeError(f'Expected {expected_images} image summaries, found {len(summary)}.')
    print('Completed run:', output_dir)
    print('Images:', len(summary))
    print('Sum of predicted detections across these images (NOT a population estimate):',
          sum(int(row['count']) for row in summary))
    show_csv(output_dir / 'summary.csv', limit=max(10, len(summary)))
    show_csv(output_dir / 'detections.csv', limit=10)
    metadata = json.loads((output_dir / 'metadata.json').read_text(encoding='utf-8'))
    figures = metadata['figures']
    if not figures:
        raise RuntimeError('The backend did not report comparison figures.')
    print(f'Displaying all {len(figures)} backend-reported comparison/detail figures:')
    for relative_name in figures:
        figure_path = (output_dir / relative_name).resolve()
        figure_path.relative_to(output_dir.resolve())
        print(relative_name)
        display(Image(filename=str(figure_path), width=1400))

show_run(PATCH_OUTPUT, expected_images=4)


## 7. Optional: both full-resolution survey images

Set `RUN_FULL_RESOLUTION = True` in the configuration cell, rerun configuration/setup/environment, then run these cells after sample preparation and the patch section. The default is **off**. The `full` selection chooses the two source images larger than 512 pixels in either dimension and processes their original 5472 × 3648 dimensions using overlapping tiles; it does not silently resize them to patches. Full-image stitching and visualization need additional memory and disk space. If resources are insufficient, keep this switch off or explicitly select OWL-C; the backend will not silently change models.

The new run has separate tables, figures, and original-pixel coordinates. Review backend-provided detail figures when available to see individual points in dense regions.

In [ ]:
FULL_OUTPUT = None
if RUN_FULL_RESOLUTION:
    FULL_OUTPUT = run_inference(SAMPLE_IMAGES_DIR, 'full', 'full')
else:
    print('Full-resolution inference skipped. Set RUN_FULL_RESOLUTION=True to opt in to both survey images.')


In [ ]:
if FULL_OUTPUT is not None:
    show_run(FULL_OUTPUT, expected_images=2)
else:
    print('No full-resolution run requested; the four-patch results remain available.')


## 8. Optional: your own image directory

Set `CUSTOM_IMAGES_DIR` to an existing directory and separately set `RUN_CUSTOM_IMAGES = True`. Use RGB-compatible PNG, JPEG (`.jpg`/`.jpeg`), or TIFF (`.tif`/`.tiff`) images; the backend selects supported image files deterministically. Custom inference uses `selection='all'`, not the sample patch/full-size filters. Put the intended inputs together in the selected directory; this is image inference, not geospatial reprojection or species classification. No annotation CSV or fake ground truth is needed.

Use only imagery you are authorized to process. Inference remains in your chosen local or hosted runtime; selecting Colab places your uploaded imagery in that hosted environment. Results use the same original-pixel coordinate and count conventions as above.

In [ ]:
CUSTOM_OUTPUT = None
if RUN_CUSTOM_IMAGES:
    if CUSTOM_IMAGES_DIR is None:
        raise ValueError('Set CUSTOM_IMAGES_DIR to your image directory before enabling RUN_CUSTOM_IMAGES.')
    custom_images = repo_path(CUSTOM_IMAGES_DIR)
    if not custom_images.is_dir():
        raise NotADirectoryError(custom_images)
    CUSTOM_OUTPUT = run_inference(custom_images, 'all', 'custom')
else:
    print('Custom-image inference skipped. Set CUSTOM_IMAGES_DIR and RUN_CUSTOM_IMAGES=True to opt in.')


In [ ]:
if CUSTOM_OUTPUT is not None:
    show_run(CUSTOM_OUTPUT)


## 9. Keep the exports

Every completed run saves `detections.csv`, `summary.csv`, `metadata.json`, comparison figures under `figures/`, and point overlays under `overlays/`. Metadata records the run settings and the actual figure filenames; the frontend reads that list instead of guessing names for long input filenames. All default artifacts stay under the ignored `demo_data/` directory. Save outputs you need before a hosted runtime is recycled.

Local Jupyter links below work when the server is launched from the repository root. VS Code users can open the printed paths directly. In Colab, use the Files panel or explicitly download an artifact in a new cell:

```python
from google.colab import files
files.download(str(PATCH_OUTPUT / 'detections.csv'))
files.download(str(PATCH_OUTPUT / 'summary.csv'))
# For optional runs, use FULL_OUTPUT or CUSTOM_OUTPUT after successful inference.
```

These commands only download existing outputs to your browser; nothing is uploaded or published automatically.

In [ ]:
for label, output_dir in [('patches', PATCH_OUTPUT), ('full', FULL_OUTPUT), ('custom', CUSTOM_OUTPUT)]:
    if output_dir is None:
        continue
    print(f'{label} exports: {output_dir}')
    metadata = json.loads((output_dir / 'metadata.json').read_text(encoding='utf-8'))
    paths = [output_dir / name for name in ['detections.csv', 'summary.csv', 'metadata.json']]
    paths += [output_dir / name for name in metadata['figures']]
    paths += sorted((output_dir / 'overlays').glob('*.png'))
    for path in paths:
        print(' ', path)
        if not IN_COLAB:
            display(FileLink(os.path.relpath(path, Path.cwd())))
if IN_COLAB:
    print('Download desired artifacts using the Files panel or files.download, as shown above.')


## Reproducibility, licenses, and limitations

- **Sample hosting:** [release owl-notebook-v1](https://github.com/microsoft/MegaDetector-Overhead/releases/tag/owl-notebook-v1) hosts the exact ZIP and accompanying checksum/attribution files. `SAMPLE_URL=''` uses the manifest URL. The ZIP stays outside Git history.
- **Sample provenance:** the contributor maps the DJI patches to [SheepCounter](https://universe.roboflow.com/riisprivate/sheepcounter) (**Public Domain**, contributor-confirmed), the two large JPGs to [HerdNet-associated data](https://dataverse.uliege.be/dataset.xhtml?persistentId=doi:10.58119/ULG/MIRUU5) (**CC BY-NC-SA 4.0**, contributor-confirmed), and the CAH patches to [the caribou release](https://zenodo.org/records/20802844) (**CC BY-NC-SA 4.0**, verified from the primary release, which requests citation of the OWL paper). Preserve the source-specific credits, license links, and applicable notices when distributing the mixed archive. `notebooks/sample_data.json` records exact per-file associations and evidence provenance; no source was inferred solely from a filename.
- **Code and model access:** use the matching notebook/backend/manifest from this repository. Both model files passed full public downloads on 2026-09-08 with published SHA-256 and size checks. Repeated fresh downloads on 2026-09-09 encountered intermittent Zenodo gateway failures and an interrupted transfer; retries and the official alternate endpoint preserve integrity, but cannot guarantee remote-service availability. The release declares CC BY-NC-SA 4.0; review that license and the included [DINOv3 terms](../dinov3/LICENSE.md).
- **Fresh environment:** on 2026-09-09, the documented locked GPU setup installed into an empty `.venv` in a separate working snapshot, with no author Conda or `PYTHONPATH` dependency. The full notebook completed there using the supplied sample archive and previously verified checkpoint after remote downloads failed. This verifies environment independence on the tested Linux/CUDA host, not every operating system or uninterrupted download availability.
- **Local backend evidence (2026-09-08):** 14 focused backend unit tests passed, including successful/corrupt download mocks. Real OWL-D CUDA inference on the four patches produced retained predicted counts 58, 19, 14, 9 (100 total), using a V100 32 GB in FP32 with 3.543 GiB peak PyTorch GPU allocation. Explicit OWL-C CPU inference produced 55, 17, 14, 9 (95 total). Real full-resolution OWL-D inference completed for both 5472 × 3648 images, using 160 tiles each, with predicted counts 4 and 6 and peak allocation 3.5815 GiB. All these real backend routes wrote output artifacts. These are backend CLI results and localization counts, not ground truth, accuracy, or a population estimate. The observed approximately 3.6 GiB GPU tensor allocation is not a universal minimum memory requirement: allocator reservations, CUDA/kernel overhead, and other memory use differ.
- **Local notebook execution verified (2026-09-08):** nbclient completed this 23-cell notebook with OWL-D on all four patches and both full-resolution images. A separate explicit OWL-C CPU notebook run completed the patch route. Notebook outputs captured environment information, progress messages, tables, and inline figures; counts matched the backend results above. No saved execution outputs are included in this source notebook. Actual Colab execution remains unverified and must be tested separately before advertising hosted support.
- **Failure handling:** on missing CUDA, fix the environment or explicitly select OWL-C; on memory exhaustion, inspect the reported resources and disable full-resolution inference or use a suitable runtime. A missing URL, checksum failure, unreadable image, or checkpoint mismatch is an error, not evidence of zero animals. Never bypass integrity checks or add fabricated annotations.

### Reproduce the verified local notebook executions

From the repository root, use [tests/execute_owl_notebook.py](../tests/execute_owl_notebook.py) through the existing project interpreter (not bare `uv run`):

```bash
.venv/bin/python tests/execute_owl_notebook.py --full \
    --output demo_data/owl_notebook/my_check_owld.ipynb
.venv/bin/python tests/execute_owl_notebook.py --model owl-c --device cpu \
    --output demo_data/owl_notebook/my_check_owlc_cpu.ipynb
```

Choose a **new output filename for every rerun**: the runner refuses to overwrite an existing executed notebook. Success prints `Executed notebook: ...` and returns exit code 0; inspect that file for progress, tables, and figures. To isolate model downloads, the runner accepts `--model-cache PATH` (choose a new empty cache). The verified [OWL-D executed notebook](../demo_data/owl_notebook/executed_owld.ipynb) and [OWL-C CPU executed notebook](../demo_data/owl_notebook/executed_owlc_cpu.ipynb) are **ignored local artifacts**, not published repository files; these links work only in a local checkout where the artifacts exist. Preserve any outputs you need before recycling a hosted runtime. The source notebook remains output-free.

See [installation](../INSTALL.md), [demo documentation](../docs/demo.md), and [dataset/access notes](../docs/datasets.md). The existing caribou evaluation shell demos remain separate and retain their original behavior.